# Relation Extraction — Sirah Nabawiyah

Menghasilkan `nodes.csv` dan `edges.csv` untuk konstruksi Knowledge Graph di Neo4j.

**Input:** `sirah_prelabelled.csv`\
**Output:** `nodes.csv` (daftar entitas unik), `edges.csv` (daftar relasi antar entitas)

---
## 1.1 Import & Konfigurasi Path

In [1]:
import re
import json
import hashlib
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ── Konfigurasi Path ─────────────────────────────────────────────────────────
BASE_DIR = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah")
IN_PRELABELLED = BASE_DIR / "data" / "result" / "manual_labelling" / "sirah_prelabelled.csv"
IN_ALIAS_MAP = BASE_DIR / "data" / "result" / "alias_clustering" / "alias_map.json"
OUT_DIR = BASE_DIR / "data" / "result" / "relation_result"
OUT_NODES = OUT_DIR / "nodes.csv"
OUT_EDGES = OUT_DIR / "edges.csv"

## 1.2 Alias Normalisasi

In [2]:
def load_alias_map(filepath: Path) -> dict:
    """Load alias map dari alias_clustering.py output."""
    if not filepath.exists():
        print(f"  WARNING: {filepath} tidak ditemukan, pakai tanpa alias")
        return {}
    data = json.loads(filepath.read_text(encoding="utf-8"))
    # Flatten: {label: {name: canonical}} → {label::name: canonical}
    flat = {}
    for label, mapping in data.items():
        for name, canonical in mapping.items():
            flat[f"{label}::{name}"] = canonical
    return flat


def normalize_name(text, label, alias_map):
    """Normalisasi nama entitas ke bentuk kanonik menggunakan alias map."""
    if not isinstance(text, str):
        return str(text).strip()

    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    key = f"{label}::{text}"
    return alias_map.get(key, text)


def generate_node_id(name, label):
    """Generate ID unik untuk node berdasarkan nama dan label."""
    key = f"{label}::{name}"
    return hashlib.md5(key.encode()).hexdigest()[:12]

## 1.3 Proximity & Context

In [3]:
def split_into_sentences(text):
    """Pecah teks menjadi kalimat berdasarkan tanda baca akhir."""
    if not isinstance(text, str):
        return []
    # Split di titik, tanda tanya, tanda seru
    # Tetapi hindari split di singkatan umum (Mr., Dr., dll)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]


def get_sentence_at_position(text, pos):
    """Dapatkan kalimat yang mengandung posisi karakter tertentu."""
    sentences = split_into_sentences(text)
    current_pos = 0
    for sent in sentences:
        sent_start = text.find(sent, current_pos)
        if sent_start == -1:
            continue
        sent_end = sent_start + len(sent)
        if sent_start <= pos < sent_end:
            return sent, sent_start, sent_end
        current_pos = sent_end
    return text, 0, len(text)


def are_in_same_context(ent1_start, ent1_end, ent2_start, ent2_end, text, max_distance=200):
    """
    Cek apakah dua entitas berada dalam konteks yang sama:
    1. Dalam kalimat yang sama, ATAU
    2. Dalam jarak < max_distance karakter
    """
    # Cek kalimat yang sama
    sent1, _, _ = get_sentence_at_position(text, ent1_start)
    sent2, _, _ = get_sentence_at_position(text, ent2_start)
    if sent1 == sent2:
        return True

    # Cek kedekatan posisi
    # Hitung jarak antara ujung terdekat kedua entitas
    distance = min(
        abs(ent1_start - ent2_end),
        abs(ent2_start - ent1_end),
    )
    return distance < max_distance


def extract_evidence(text, ent1_start, ent1_end, ent2_start, ent2_end, context_window=50):
    """Ekstrak cuplikan teks yang mengandung kedua entitas sebagai bukti relasi."""
    start = max(0, min(ent1_start, ent2_start) - context_window)
    end = min(len(text), max(ent1_end, ent2_end) + context_window)

    evidence = text[start:end].strip()
    # Tambahkan ellipsis jika terpotong
    if start > 0:
        evidence = "..." + evidence
    if end < len(text):
        evidence = evidence + "..."

    return evidence

## 1.4 Contextual Guards

Menyaring false positive dari proximity-based extraction dengan mengecek evidence teks.

In [4]:
# Perawi/penulis kitab yang BUKAN tokoh sejarah Sirah
KNOWN_NARRATORS = {
    "Ibnu Hisyam", "Ibnu Ishaq", "Ath-Thabari", "An-Nawawi",
    "Ibnul Qayyim", "Al-Khadhri", "Abu Dawud", "Ibnu Sa'd",
    "Al-Waqidi",
}

# Pola-pola INVALID untuk OCCURRED_AT (lokasi bukan tempat kejadian)
_OCCURRED_AT_INVALID_PATTERNS = [
    r"wakil\s+(?:beliau\s+)?di\s+{loc}",
    r"(?:kembali|pulang)\s+(?:lagi\s+)?ke\s+{loc}",
    r"kepulangan\s+.*?ke\s+{loc}",
    r"keluar\s+dari\s+{loc}",
    r"penduduk\s+{loc}",
    r"(?:berangkat|berasal)\s+dari\s+(?:\w+\s+){{0,2}}{loc}",
    r"berpencar\s+ke\s+{loc}",
    r"datang\s+ke\s+{loc}",
    r"pergi\s+(?:.*?\s+)?ke\s+{loc}",
    r"di\s+{loc}\s+dulu",
    r"orang-orang\s+(?:musyrik|kafir|munafik)\s+{loc}",
    r"raja-raja\s+{loc}",
    r"(?:tinggal|menetap)\s+(?:.*?\s+)?di\s+{loc}",
    r"membeli\s+.*?di\s+{loc}",
    r"menyerang\s+(?:pinggiran\s+)?{loc}",
    r"(?:harta\s+rampasan|ghanimah).*?(?:adalah|yaitu)\s+{loc}",
    r"(?:seluruh|di\s+seluruh)\s+{loc}",
    r"antara\s+\w[\w\s]*?dan\s+{loc}",
    r"(?:masuk|melarikan\s+diri|pelarian.*?masuk)\s+ke\s+{loc}",
    r"menuju\s+(?:ke\s+)?{loc}",
    r"(?:langit|ufuk)\s+{loc}",
    r"surat\s+dari\s+.*?{loc}",
    r"bergabung\s+dengan\s+{loc}",
    r"tiba\s+di\s+{loc}",
    r"sedang\s+berada\s+di\s+{loc}",
    r"sepulang\s+(?:.*?\s+)?dari\s+{loc}",
    r"sekembalinya\s+dari\s+{loc}",
    r"perjalanan\s+pulang\s+ke\s+{loc}",
    r"bergerak\s+ke\s+arah\s+.*?{loc}",
    r"(?:setelah|sejak|sebelum)\s+(?:perang\s+)?(?:penaklukan|penaklukkan)\s+{loc}",
    r"yang\s+berada\s+di\s+{loc}",
    r"(?:dalam\s+)?(?:penaklukkan|penaklukan)\s+{loc}",
]

# Pola-pola INVALID untuk OCCURRED_ON (waktu bukan waktu event)
_OCCURRED_ON_INVALID_PATTERNS = [
    r"(?:kembali|pulang|kepulangan)\s+.*?(?:pada|akhir)\s+.*?{time}",
    r"meninggal\s+(?:dunia\s+)?(?:pada|di)\s+.*?{time}",
    r"(?:setelah|seusai|sesudah|sepulang)\s+.*?(?:pada|di)\s+.*?{time}",
]


def is_invalid_involved_in(person_name, evidence):
    """Cek apakah PERSON benar-benar terlibat dalam EVENT.
    Returns True jika relasi INVALID (harus di-skip)."""

    # 1. Known narrators/penulis kitab
    if person_name in KNOWN_NARRATORS:
        return True

    # 2. Referensi ayat Al-Quran: "(Yusuf: 90)"
    if re.search(r"\(\s*" + re.escape(person_name) + r"\s*:\s*\d+\s*\)", evidence):
        return True

    # 3. Disebutkan meninggal dunia (bukan partisipasi event)
    if re.search(re.escape(person_name) + r"\s+(?:ini\s+)?meninggal\s+dunia", evidence):
        return True

    return False


# Variasi ejaan lokasi yang sering muncul dalam teks
_SPELLING_VARIANTS = {
    "Yatsrib": ["Yastrib"],
    "Yastrib": ["Yatsrib"],
}


def _get_loc_variants(loc_name):
    """Dapatkan variasi ejaan lokasi."""
    variants = [loc_name]
    if loc_name in _SPELLING_VARIANTS:
        variants.extend(_SPELLING_VARIANTS[loc_name])
    return variants


def is_invalid_occurred_at(event_name, loc_name, evidence):
    """Cek apakah LOCATION benar-benar tempat kejadian EVENT.
    Returns True jika relasi INVALID (harus di-skip)."""

    # Nama event mengandung lokasi → selalu valid
    if loc_name.lower() in event_name.lower():
        return False

    # Event adalah penaklukan lokasi ini → valid
    if re.search(r"(?:Fath|fath|penakluk)", event_name, re.IGNORECASE):
        for loc_v in _get_loc_variants(loc_name):
            loc_esc = re.escape(loc_v)
            if re.search(rf"(?:penaklukkan|penaklukan|menaklukkan)\s+{loc_esc}",
                         evidence, re.IGNORECASE):
                return False

    # Cek pola-pola invalid (coba semua variasi ejaan)
    for loc_v in _get_loc_variants(loc_name):
        loc_esc = re.escape(loc_v)
        for pattern in _OCCURRED_AT_INVALID_PATTERNS:
            pat = pattern.replace("{loc}", loc_esc)
            if re.search(pat, evidence, re.IGNORECASE):
                return True

    return False


def is_invalid_occurred_on(event_name, time_name, evidence):
    """Cek apakah TIME benar-benar waktu EVENT.
    Returns True jika relasi INVALID (harus di-skip)."""
    time_esc = re.escape(time_name)
    for pattern in _OCCURRED_ON_INVALID_PATTERNS:
        pat = pattern.replace("{time}", time_esc)
        if re.search(pat, evidence, re.IGNORECASE):
            return True
    return False

## 1.5 Load & Prepare Data

In [5]:
def load_and_prepare_data(filepath, alias_map):
    """Load CSV pre-labelled dan siapkan data."""
    df = pd.read_csv(filepath, sep=";", encoding="utf-8-sig")
    df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

    # Filter baris yang punya label valid
    valid_labels = {"PERSON", "EVENT", "LOCATION", "TIME"}
    df = df[df["label"].isin(valid_labels)].copy()

    # Konversi start_char dan end_char ke integer
    df["start_char"] = pd.to_numeric(df["start_char"], errors="coerce").fillna(0).astype(int)
    df["end_char"] = pd.to_numeric(df["end_char"], errors="coerce").fillna(0).astype(int)

    # Normalisasi nama entitas menggunakan alias map
    df["canonical_name"] = df.apply(
        lambda row: normalize_name(row["entity_text"], row["label"], alias_map), axis=1
    )

    # Statistik alias resolution
    n_resolved = (df["entity_text"] != df["canonical_name"]).sum()

    print(f"Data loaded: {len(df)} entitas valid")
    print(f"Alias resolved: {n_resolved} entitas")
    print(f"Distribusi label:")
    for label, count in df["label"].value_counts().items():
        n_unique = df[df["label"] == label]["canonical_name"].nunique()
        print(f"  {label}: {count} mentions, {n_unique} unique (setelah alias)")

    return df

## 1.6 Build Relations (Proximity-based)

In [6]:
def build_relations(df):
    """Bentuk relasi antar entitas berdasarkan chunk."""
    relations = []
    chunks_with_events = 0
    chunks_without_events = 0

    # Kelompokkan entitas per chunk
    grouped = df.groupby("chunk_id")

    for chunk_id, chunk_df in grouped:
        # Ambil teks chunk (sama untuk semua baris di chunk)
        chunk_text = chunk_df.iloc[0].get("teks_chunk", "")
        halaman = chunk_df.iloc[0].get("halaman", "")

        # Pisahkan entitas berdasarkan label
        events = chunk_df[chunk_df["label"] == "EVENT"]
        persons = chunk_df[chunk_df["label"] == "PERSON"]
        locations = chunk_df[chunk_df["label"] == "LOCATION"]
        times = chunk_df[chunk_df["label"] == "TIME"]

        if len(events) == 0:
            chunks_without_events += 1
            continue

        chunks_with_events += 1

        for _, event in events.iterrows():
            ev_start = event["start_char"]
            ev_end = event["end_char"]
            ev_name = event["canonical_name"]

            # PERSON → EVENT (INVOLVED_IN)
            for _, person in persons.iterrows():
                p_start = person["start_char"]
                p_end = person["end_char"]

                if are_in_same_context(p_start, p_end, ev_start, ev_end, chunk_text):
                    evidence = extract_evidence(
                        chunk_text, p_start, p_end, ev_start, ev_end
                    )
                    # Guard: filter perawi, ayat, kematian
                    if is_invalid_involved_in(person["canonical_name"], evidence):
                        continue
                    relations.append({
                        "source_name": person["canonical_name"],
                        "source_label": "PERSON",
                        "relation_type": "INVOLVED_IN",
                        "target_name": ev_name,
                        "target_label": "EVENT",
                        "chunk_id": chunk_id,
                        "evidence": evidence,
                        "halaman": halaman,
                    })

            # EVENT → LOCATION (OCCURRED_AT)
            for _, location in locations.iterrows():
                l_start = location["start_char"]
                l_end = location["end_char"]

                if are_in_same_context(ev_start, ev_end, l_start, l_end, chunk_text):
                    evidence = extract_evidence(
                        chunk_text, ev_start, ev_end, l_start, l_end
                    )
                    # Guard: filter lokasi yang bukan tempat kejadian
                    if is_invalid_occurred_at(ev_name, location["canonical_name"], evidence):
                        continue
                    relations.append({
                        "source_name": ev_name,
                        "source_label": "EVENT",
                        "relation_type": "OCCURRED_AT",
                        "target_name": location["canonical_name"],
                        "target_label": "LOCATION",
                        "chunk_id": chunk_id,
                        "evidence": evidence,
                        "halaman": halaman,
                    })

            # EVENT → TIME (OCCURRED_ON)
            for _, time in times.iterrows():
                t_start = time["start_char"]
                t_end = time["end_char"]

                if are_in_same_context(ev_start, ev_end, t_start, t_end, chunk_text):
                    evidence = extract_evidence(
                        chunk_text, ev_start, ev_end, t_start, t_end
                    )
                    # Guard: filter waktu yang bukan waktu event
                    if is_invalid_occurred_on(ev_name, time["canonical_name"], evidence):
                        continue
                    relations.append({
                        "source_name": ev_name,
                        "source_label": "EVENT",
                        "relation_type": "OCCURRED_ON",
                        "target_name": time["canonical_name"],
                        "target_label": "TIME",
                        "chunk_id": chunk_id,
                        "evidence": evidence,
                        "halaman": halaman,
                    })

    print(f"\nChunks dengan EVENT: {chunks_with_events}")
    print(f"Chunks tanpa EVENT : {chunks_without_events}")
    print(f"Total relasi mentah: {len(relations)}")

    return relations

## 1.7 Deduplikasi Relasi

In [7]:
def deduplicate_relations(relations):
    """
    Deduplikasi relasi:
    Jika source, target, dan relation_type sama, gabungkan evidence dan chunk_id.
    """
    dedup = {}

    for rel in relations:
        key = (rel["source_name"], rel["source_label"],
               rel["relation_type"],
               rel["target_name"], rel["target_label"])

        if key not in dedup:
            dedup[key] = {
                "source_name": rel["source_name"],
                "source_label": rel["source_label"],
                "relation_type": rel["relation_type"],
                "target_name": rel["target_name"],
                "target_label": rel["target_label"],
                "chunk_ids": [rel["chunk_id"]],
                "evidences": [rel["evidence"]],
                "halaman": [str(rel["halaman"])],
            }
        else:
            if rel["chunk_id"] not in dedup[key]["chunk_ids"]:
                dedup[key]["chunk_ids"].append(rel["chunk_id"])
                dedup[key]["evidences"].append(rel["evidence"])
            if str(rel["halaman"]) not in dedup[key]["halaman"]:
                dedup[key]["halaman"].append(str(rel["halaman"]))

    # Flatten ke list of dicts
    result = []
    for key, val in dedup.items():
        result.append({
            "source_name": val["source_name"],
            "source_label": val["source_label"],
            "relation_type": val["relation_type"],
            "target_name": val["target_name"],
            "target_label": val["target_label"],
            "chunk_id": " | ".join(val["chunk_ids"]),
            "evidence": val["evidences"][0],  # Ambil evidence pertama saja
            "halaman": " | ".join(val["halaman"]),
            "frequency": len(val["chunk_ids"]),
        })

    print(f"Relasi setelah deduplikasi: {len(result)}")
    return result

## 1.8 Build Nodes

In [8]:
def build_nodes(df):
    """Bangun daftar node unik dari entitas."""
    node_map = {}

    for _, row in df.iterrows():
        name = row["canonical_name"]
        label = row["label"]
        original = row["entity_text"]
        chunk_id = row["chunk_id"]

        key = (label, name)

        if key not in node_map:
            node_map[key] = {
                "node_id": generate_node_id(name, label),
                "name": name,
                "label": label,
                "aliases": set(),
                "chunk_ids": set(),
                "frequency": 0,
            }

        node_map[key]["frequency"] += 1
        node_map[key]["chunk_ids"].add(str(chunk_id))
        if original != name:
            node_map[key]["aliases"].add(original)

    # Convert sets ke strings
    result = []
    for key, val in node_map.items():
        result.append({
            "node_id": val["node_id"],
            "name": val["name"],
            "label": val["label"],
            "aliases": " | ".join(sorted(val["aliases"])) if val["aliases"] else "",
            "chunk_ids": " | ".join(sorted(val["chunk_ids"])),
            "frequency": val["frequency"],
        })

    # Sort by label lalu frequency descending
    result.sort(key=lambda x: (x["label"], -x["frequency"]))

    print(f"\nTotal node unik: {len(result)}")
    for label in ["PERSON", "EVENT", "LOCATION", "TIME"]:
        count = sum(1 for n in result if n["label"] == label)
        print(f"  {label}: {count}")

    return result

## 1.9 Print Sample

In [9]:
def print_sample_relations(edges_df, n=10):
    """Tampilkan sample relasi untuk verifikasi."""
    print(f"\n{'='*80}")
    print(f"SAMPLE {n} RELASI:")
    print(f"{'='*80}")

    for rel_type in ["INVOLVED_IN", "OCCURRED_AT", "OCCURRED_ON"]:
        subset = edges_df[edges_df["relation_type"] == rel_type].head(n)
        if len(subset) == 0:
            continue
        print(f"\n-- {rel_type} ({len(edges_df[edges_df['relation_type'] == rel_type])} total) --")
        for _, row in subset.iterrows():
            src = f"[{row['source_label']}] {row['source_name']}"
            tgt = f"[{row['target_label']}] {row['target_name']}"
            print(f"  {src} --> {rel_type} --> {tgt}")
            # Tampilkan evidence yang dipersingkat
            ev = row["evidence"][:100] + "..." if len(str(row["evidence"])) > 100 else row["evidence"]
            print(f"    Evidence: {ev}")
            print()

---
## 1.10 Jalankan Pipeline

In [10]:
print("=" * 60)
print("RELATION EXTRACTION - Sirah Nabawiyah")
print("=" * 60)

# 1. Load data & alias map
print("\n[1/5] Loading data...")
alias_map = load_alias_map(IN_ALIAS_MAP)
print(f"  Alias map: {len(alias_map)} entries loaded")
df = load_and_prepare_data(IN_PRELABELLED, alias_map)

# 2. Build nodes
print("\n[2/5] Building node list...")
nodes = build_nodes(df)

# 3. Build relations
print("\n[3/5] Building relations...")
relations_raw = build_relations(df)

# 4. Deduplicate
print("\n[4/5] Deduplicating relations...")
relations_dedup = deduplicate_relations(relations_raw)

# 5. Save output
print("\n[5/5] Saving output...")
OUT_DIR.mkdir(parents=True, exist_ok=True)

nodes_df = pd.DataFrame(nodes)
nodes_df.to_csv(OUT_NODES, index=False, sep=";", encoding="utf-8-sig")
print(f"  Nodes saved to: {OUT_NODES}")

edges_df = pd.DataFrame(relations_dedup)
edges_df.to_csv(OUT_EDGES, index=False, sep=";", encoding="utf-8-sig")
print(f"  Edges saved to: {OUT_EDGES}")

# 6. Statistik akhir
print(f"\n{'='*60}")
print("RINGKASAN STATISTIK")
print(f"{'='*60}")
print(f"Total node unik    : {len(nodes_df)}")
print(f"Total edge unik    : {len(edges_df)}")
if len(edges_df) > 0:
    print(f"\nDistribusi relasi:")
    for rel_type, count in edges_df["relation_type"].value_counts().items():
        print(f"  {rel_type}: {count}")

# 7. Sample relasi
if len(edges_df) > 0:
    print_sample_relations(edges_df, n=5)

print(f"\n{'='*60}")
print("SELESAI!")
print(f"{'='*60}")

RELATION EXTRACTION - Sirah Nabawiyah

[1/5] Loading data...
  Alias map: 145 entries loaded
Data loaded: 6000 entitas valid
Alias resolved: 1665 entitas
Distribusi label:
  PERSON: 4068 mentions, 658 unique (setelah alias)
  LOCATION: 1434 mentions, 51 unique (setelah alias)
  TIME: 307 mentions, 147 unique (setelah alias)
  EVENT: 191 mentions, 41 unique (setelah alias)

[2/5] Building node list...

Total node unik: 897
  PERSON: 658
  EVENT: 41
  LOCATION: 51
  TIME: 147

[3/5] Building relations...

Chunks dengan EVENT: 133
Chunks tanpa EVENT : 665
Total relasi mentah: 358

[4/5] Deduplicating relations...
Relasi setelah deduplikasi: 226

[5/5] Saving output...
  Nodes saved to: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\relation_result\nodes.csv
  Edges saved to: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\relation_result\edges.csv

RINGKASAN STATISTIK
Total node unik    : 897
Total edge unik    : 226

Distri